# notebook 05 — kpis and export

calculate the final kpis, build summary tables, and export everything to csv files ready for tableau.

**input:** data/processed/cleaned_sales.csv, data/processed/cancellations.csv  
**output:** tableau_ready.csv, kpi_summary.csv, monthly_summary.csv, country_summary.csv, product_summary.csv, customer_summary.csv

In [1]:
# Import libraries and load data
import pandas as pd
import numpy as np

# Load clean sales and cancellations
df = pd.read_csv('../data/processed/cleaned_sales.csv')
cancellations_df = pd.read_csv('../data/processed/cancellations.csv')


In [2]:
# 1. Compute KPIs
total_revenue = df['Revenue'].sum()
total_orders = df['InvoiceNo'].nunique()
total_products_sold = df['Quantity'].sum()
avg_order_value = total_revenue / total_orders

clean_rows = len(df)
cancellation_rows = len(cancellations_df)
cancellation_rate = (cancellation_rows / (clean_rows + cancellation_rows)) * 100

unique_customers = df['CustomerID'].nunique()
unique_products = df['StockCode'].nunique()

uk_revenue = df[df['is_uk'] == True]['Revenue'].sum()
intl_revenue = df[df['is_uk'] == False]['Revenue'].sum()

# Best and worst month by revenue
monthly_rev = df.groupby(['Year', 'Month'])['Revenue'].sum().reset_index()
best_month_row = monthly_rev.loc[monthly_rev['Revenue'].idxmax()]
worst_month_row = monthly_rev.loc[monthly_rev['Revenue'].idxmin()]

best_month_str = f"{int(best_month_row['Year'])}-{str(int(best_month_row['Month'])).zfill(2)}"
worst_month_str = f"{int(worst_month_row['Year'])}-{str(int(worst_month_row['Month'])).zfill(2)}"

print("=== FINAL KPIs ===")
print(f"1. Total Revenue: £{total_revenue:,.2f}")
print(f"2. Total Orders: {total_orders:,}")
print(f"3. Total Products Sold: {total_products_sold:,}")
print(f"4. Average Order Value: £{avg_order_value:,.2f}")
print(f"5. Cancellation Rate: {cancellation_rate:.2f}%")
print(f"6. Number of Unique Customers: {unique_customers:,}")
print(f"7. Number of Unique Products: {unique_products:,}")
print(f"8. Revenue from UK: £{uk_revenue:,.2f} | International: £{intl_revenue:,.2f}")
print(f"9. Best Month by Revenue: {best_month_str} (£{best_month_row['Revenue']:,.2f})")
print(f"10. Worst Month by Revenue: {worst_month_str} (£{worst_month_row['Revenue']:,.2f})")


=== FINAL KPIs ===
1. Total Revenue: £10,254,128.44
2. Total Orders: 19,776
3. Total Products Sold: 5,561,480
4. Average Order Value: £518.51
5. Cancellation Rate: 1.74%
6. Number of Unique Customers: 4,334
7. Number of Unique Products: 3,908
8. Revenue from UK: £8,725,776.79 | International: £1,528,351.65
9. Best Month by Revenue: 2011-11 (£1,453,262.69)
10. Worst Month by Revenue: 2011-02 (£508,005.00)


In [3]:
# 2. Feature Engineering for Tableau
# Customer Lifetime Value (LTV)
df['Customer_LTV'] = df.groupby('CustomerID')['Revenue'].transform('sum')

# Cancellation Count per Customer
cancel_counts = cancellations_df.groupby('CustomerID')['InvoiceNo'].nunique().reset_index()
cancel_counts.columns = ['CustomerID', 'Customer_Cancellation_Count']

# Merge back
df = pd.merge(df, cancel_counts, on='CustomerID', how='left')
df['Customer_Cancellation_Count'] = df['Customer_Cancellation_Count'].fillna(0)

print("Engineered Customer_LTV and Customer_Cancellation_Count added to main dataset.")


Engineered Customer_LTV and Customer_Cancellation_Count added to main dataset.


In [4]:
# 3. Create Aggregated Tables

# Monthly summary: Year, Month, Revenue, Order_Count, Avg_Order_Value
monthly_summary = df.groupby(['Year', 'Month']).agg(
    Revenue=('Revenue', 'sum'),
    Order_Count=('InvoiceNo', 'nunique')
).reset_index()
monthly_summary['Avg_Order_Value'] = monthly_summary['Revenue'] / monthly_summary['Order_Count']

# Country summary: Country, Revenue, Order_Count, Avg_Order_Value
country_summary = df.groupby('Country').agg(
    Revenue=('Revenue', 'sum'),
    Order_Count=('InvoiceNo', 'nunique')
).reset_index()
country_summary['Avg_Order_Value'] = country_summary['Revenue'] / country_summary['Order_Count']

# Product summary: Description, StockCode, Revenue, Units_Sold, Avg_Unit_Price
product_summary = df.groupby(['StockCode', 'Description']).agg(
    Revenue=('Revenue', 'sum'),
    Units_Sold=('Quantity', 'sum'),
    Avg_Unit_Price=('UnitPrice', 'mean')
).reset_index()

# Customer CLV table: CustomerID, Country, Total_Revenue, Order_Count, Cancellation_Count
# Getting the first country for each customer since they generally don't change
customer_country = df.groupby('CustomerID')['Country'].first().reset_index()
customer_summary = df.groupby('CustomerID').agg(
    Total_Revenue=('Revenue', 'sum'),
    Order_Count=('InvoiceNo', 'nunique')
).reset_index()
customer_summary = pd.merge(customer_summary, customer_country, on='CustomerID', how='left')
customer_summary = pd.merge(customer_summary, cancel_counts, on='CustomerID', how='left')
customer_summary['Customer_Cancellation_Count'] = customer_summary['Customer_Cancellation_Count'].fillna(0)

print("Aggregated tables generated successfully.")


Aggregated tables generated successfully.


In [5]:
# 4. Save Outputs for Tableau

# Main dataset
df.to_csv('../data/processed/tableau_ready.csv', index=False)

# KPI Summary
kpi_dict = {
    'Total_Revenue': total_revenue,
    'Total_Orders': total_orders,
    'Total_Products_Sold': total_products_sold,
    'Avg_Order_Value': avg_order_value,
    'Cancellation_Rate_Pct': cancellation_rate,
    'Unique_Customers': unique_customers,
    'Unique_Products': unique_products,
    'UK_Revenue': uk_revenue,
    'Intl_Revenue': intl_revenue,
    'Best_Month': best_month_str,
    'Worst_Month': worst_month_str
}
kpi_df = pd.DataFrame([kpi_dict])
kpi_df.to_csv('../data/processed/kpi_summary.csv', index=False)

# Aggregated tables
monthly_summary.to_csv('../data/processed/monthly_summary.csv', index=False)
country_summary.to_csv('../data/processed/country_summary.csv', index=False)
product_summary.to_csv('../data/processed/product_summary.csv', index=False)
customer_summary.to_csv('../data/processed/customer_summary.csv', index=False)

print("All 6 files successfully exported to data/processed/")


All 6 files successfully exported to data/processed/


Final Dataset Overview — What Was Exported

After completing all cleaning, feature engineering, and aggregation steps, the following files are now ready in the data/processed folder for Tableau.

tableau_ready.csv
Full flat file with one row per line item. Includes all original columns plus all engineered columns: Revenue, Customer LTV, and Customer Cancellation Count.

kpi_summary.csv
A single row file containing all top-level KPIs: Total Revenue, Total Orders, Average Order Value, Cancellation Rate percentage.

monthly_summary.csv
Revenue aggregated by Year and Month. Columns: Year, Month, Revenue, Order Count, Average Order Value.

country_summary.csv
Revenue aggregated by Country. Columns: Country, Revenue, Order Count, Average Order Value.

product_summary.csv
Revenue aggregated by StockCode and Description. Columns: Description, StockCode, Revenue, Units Sold, Average Unit Price.

customer_summary.csv
One row per customer showing their full lifetime value and cancellation history. Columns: CustomerID, Country, Total Revenue, Order Count, Customer Cancellation Count.

Engineered columns added in this notebook:

Customer_LTV: Sum of all Revenue for each CustomerID. Used for customer segmentation in Tableau.
Customer_Cancellation_Count: Number of cancelled invoices per CustomerID. Used to flag high-risk or high-churn customers.

Tableau tip: Connect tableau_ready.csv as the primary data source. Use kpi_summary.csv for the KPI tiles sheet. Use the summary files for dimension-level charts like the country map, product bar chart, and monthly trend line.